# Project 36 — LangGraph Procurement Review Flow
## Compare vendor options and generate procurement recommendation

**Stack:** LangGraph · Ollama · Jupyter

**Workflow:** Gather Requirements → Compare Vendors → Score Vendors → Generate Recommendation

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

class WorkflowState(TypedDict):
    input_data: str
    accumulated: str
    final_output: str

## Step 2 — Define Workflow Nodes

In [ ]:
def gather_requirements(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 1 of a LangGraph Procurement Review Flow pipeline. "
         "Your role: gather requirements. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[gather_requirements]: " + result
    print(f"  Step 1/4: gather_requirements")
    return {"accumulated": accumulated, "final_output": result}

def compare_vendors(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 2 of a LangGraph Procurement Review Flow pipeline. "
         "Your role: compare vendors. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[compare_vendors]: " + result
    print(f"  Step 2/4: compare_vendors")
    return {"accumulated": accumulated, "final_output": result}

def score_vendors(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 3 of a LangGraph Procurement Review Flow pipeline. "
         "Your role: score vendors. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[score_vendors]: " + result
    print(f"  Step 3/4: score_vendors")
    return {"accumulated": accumulated, "final_output": result}

def generate_recommendation(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 4 of a LangGraph Procurement Review Flow pipeline. "
         "Your role: generate recommendation. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[generate_recommendation]: " + result
    print(f"  Step 4/4: generate_recommendation")
    return {"accumulated": accumulated, "final_output": result}

## Step 3 — Build and Compile Graph

In [ ]:
graph = StateGraph(WorkflowState)
graph.add_node("gather_requirements", gather_requirements)
graph.add_node("compare_vendors", compare_vendors)
graph.add_node("score_vendors", score_vendors)
graph.add_node("generate_recommendation", generate_recommendation)

graph.set_entry_point("gather_requirements")
graph.add_edge("gather_requirements", "compare_vendors")
graph.add_edge("compare_vendors", "score_vendors")
graph.add_edge("score_vendors", "generate_recommendation")
graph.add_edge("generate_recommendation", END)

app = graph.compile()
print("LangGraph Procurement Review Flow — workflow compiled!")

## Step 4 — Run the Workflow

In [ ]:
sample_input = "Analyze and process this request through the LangGraph Procurement Review Flow pipeline."
result = app.invoke({
    "input_data": sample_input, "accumulated": "", "final_output": "",
})

print("=== WORKFLOW RESULT ===")
print(result["final_output"])
print("\n=== FULL TRACE ===")
print(result["accumulated"][:1000])

## What You Learned
- **Multi-node LangGraph workflow** for compare vendor options and generate procurement recommendation
- **Progressive context accumulation** across steps
- **Structured pipeline execution** with trace logging